# 05 · Ticket clock and fields

This notebook replicates the Ticket clock and fields paragraph of the report's Results section.

> Ticket clock and fields: Opening times never step backwards in case-number order (0 of 7,470 steps) but would 1,205–1,312 times on hospital-local clocks, so case times use one U.S. Eastern clock, as IPA confirmed. Trouble Code and Category are filled on 1 of 64 open tickets and on nearly all of the 461 August 2026 tickets of any kind with status Trouble Closed. Category is set near close, and the export holds only each field's final value.

**How it works.** We count the steps between consecutive tickets in case-number order, the steps that go backwards on the export clock and on hospital-local clocks, and the open and August 2026 Trouble Closed tickets that carry a Trouble Code and a Category, in the tables that `timestamp_semantics.py` and `case_order.py` build from IPA's case export. With `RERUN = True`, the notebook re-runs those scripts in a scratch copy of the outputs and compares each file they write with the committed one; with `RERUN = False`, it reads the committed outputs. Either way, it recomputes each number from the tables in `OUT` and checks it against the report. The one exception is the lower end of the report's range of backward steps on hospital-local clocks (1,205), which came from a reviewer's zone map that is not in the repository (see below).

**Scripts and inputs.** The notebook re-runs `verification/timestamp_semantics.py` (about 3 minutes), which writes the `ts_*.csv` tables at the top of `verification/outputs/`, then `verification/case_order.py` (a few seconds), which writes the creation-order test to `case_order_steps.csv`. It reads `case_order_steps.csv` and three of the `ts_*` tables: `ts_facts.csv`, `ts_field_fill_rates.csv`, and `ts_text_markers_by_month.csv`. `docs/04_timestamp_semantics.md` (§2.1 and §5) describes the analysis. We use these committed outputs as inputs without re-running them:
- `fu/fu_ticket_last10.csv` from `db_followup.py`, because it needs the clearView telemetry database, which the notebooks do not re-read. It is a per-ticket table of the 6,113 tickets matched to a clearView device, and we use it for cross-checks and for one ticket's fields.
- For cross-checks only, `audit_case_status_by_month.csv`, `audit_case_service_group_by_status.csv`, and `audit_case_missing_category_composition.csv` from `audit_case_logic.py`. We do not re-run that script because it also reads a text dump of the midterm notebook from the gitignored `data/derived/` folder, and a test re-run without that file stopped there with an error.

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    R.rerun(OUT, 'timestamp_semantics.py')
    R.rerun(OUT, 'case_order.py')
R.compare(OUT)

timestamp_semantics.py: exit status 0, 204 s
    Done.


case_order.py: exit status 0, 3 s
    6/6 checks pass
43 files written: 43 identical.


,file written by the re-run,compared with the committed file
0,case_order_steps.csv,identical
1,ts_anchor_correction_distribution.csv,identical
2,ts_clock_hypothesis_scorecard.csv,identical
3,ts_close_duration_bins.csv,identical
4,ts_close_duration_quantiles.csv,identical
5,ts_close_hour_by_zone.csv,identical
6,ts_desk_edge_by_month.csv,identical
7,ts_dispatch_by_record_type.csv,identical
8,ts_dispatch_vs_outcome.csv,identical
9,ts_dst_desk_edge.csv,identical


## Steps in case-number order

Salesforce numbers cases in creation order, and the pseudonyms T00001 to T07471 follow the sorted 8-digit case numbers (`common.py`), so the steps are the pairs of consecutive tickets in that order, one fewer than the tickets. We count the tickets by opening month in `ts_text_markers_by_month.csv` and cross-check the total with the rows `timestamp_semantics.py` loads (`ts_facts.csv`) and with the open and closed groups of `ts_field_fill_rates.csv`.

In [3]:
facts = R.read_csv(OUT, 'ts_facts.csv')
fact = {(section, item): value for section, item, value in facts[['section', 'item', 'value']].itertuples(index=False)}
fill = R.read_csv(OUT, 'ts_field_fill_rates.csv').set_index('group')
tickets_by_month = R.read_csv(OUT, 'ts_text_markers_by_month.csv').set_index('month')['tickets']

n_tickets = int(tickets_by_month.sum())
assert n_tickets == int(fact['load', 'rows'])
assert n_tickets == fill.loc['open (Status=Dispatch)', 'tickets'] + fill.loc['closed (all other statuses)', 'tickets']
n_steps = n_tickets - 1
print(f'{n_tickets:,} tickets opened from {tickets_by_month.index[0]} to {tickets_by_month.index[-1]} make '
      f'{n_steps:,} steps between consecutive tickets in case-number order.')

7,471 tickets opened from 2025-08 to 2026-08 make 7,470 steps between consecutive tickets in case-number order.


## Backward steps: the creation-order test

`case_order.py` sorts the 7,471 tickets by case number and counts the steps between consecutive tickets that go backwards in time. On one shared clock, opening times can tie, since they are minute-precise, but cannot go backwards. For hospital-local clocks, the script shows each recorded time as the hospital's own zone would, so a Pacific ticket opened minutes after an Eastern one shows about 3 hours earlier. It also works the other way round, reading each recorded time as the hospital's local time and moving it to U.S. Eastern. It does both under two maps from hospital location to time zone: `timestamp_semantics.py`'s (the state, with 3-digit ZIP rules for the states that span two zones, in `verification/zones.py`) and the state alone. Before counting, it checks that its tickets by state and zone equal `timestamp_semantics.py`'s `ts_zone_mapping.csv`.

In [4]:
order = R.read_csv(OUT, 'case_order_steps.csv')
export, local = order.iloc[0], order.iloc[1:]
assert export.clock == 'export clock, as recorded' and export.steps == n_steps and (local.steps == n_steps).all()
primary = local[local.clock.str.startswith('shown locally') & local.zone_map.str.startswith('state + ZIP3')].iloc[0]
print(f'Export clock: {export.backward} of {export.steps:,} steps go backwards, and {export.ties} stay in the same '
      f'minute. Hospital-local clocks: {R.span(local.backward.min(), local.backward.max())} backward steps, '
      f"{primary.backward:,} on timestamp_semantics.py's map with each time shown locally.")
order

Export clock: 0 of 7,470 steps go backwards, and 173 stay in the same minute. Hospital-local clocks: 1,210–1,312 backward steps, 1,312 on timestamp_semantics.py's map with each time shown locally.


,clock,zone_map,steps,backward,ties
0,"export clock, as recorded",NaN,7470,0,173
1,shown locally: each recorded time as its hospi...,"state + ZIP3: the state, with 3-digit ZIP rule...",7470,1312,133
2,shown locally: each recorded time as its hospi...,state only: the state alone,7470,1295,134
3,read as local: each recorded time read as its ...,"state + ZIP3: the state, with 3-digit ZIP rule...",7470,1220,127
4,read as local: each recorded time read as its ...,state only: the state alone,7470,1210,128


**The lower end of the report's range is not reproduced.** The report's 1,205–1,312 comes from docs/04 §2.1. The upper end, 1,312, is the count on `timestamp_semantics.py`'s map with each recorded time shown locally. The lower end, 1,205, came from a second-wave reviewer's own zone map. docs/04 lists that review's results in `data/derived/workflowB_result.json`, which is gitignored, and its code was never committed. The maps in the repository give 1,210–1,312, so every one of them gives more than 1,200 backward steps, as docs/04 says. The check has rows for 0 and 1,312, and this table lists the lower end.

In [5]:
pd.DataFrame([{'Finding': 'Backward steps on hospital-local clocks, lower end', 'Report': '1,205',
               'Lowest count on the maps in the repository': R.num(local.backward.min()),
               'Reproduced': "no: the reviewer's zone map is not in the repository"}])

,Finding,Report,Lowest count on the maps in the repository,Reproduced
0,"Backward steps on hospital-local clocks, lower...","1,205","1,210",no: the reviewer's zone map is not in the repo...


## One U.S. Eastern clock

That the shared clock is U.S. Eastern time is IPA's answer at the Oct 13th meeting, recorded in docs/04 (update of Oct 14th, and question 1 in §6). It is a record of IPA's answer, so the check has no row for it.

## Open tickets with a Trouble Code and a Category

An open ticket has status Dispatch at export, and `ts_field_fill_rates.csv` gives the open tickets' count and the share of them that carry each field; the shares have four decimals, which pins down the count in any group under 5,000 tickets. We cross-check with `timestamp_semantics.py`'s own count of open tickets with a Trouble Code (`ts_facts.csv`), with `audit_case_logic.py`'s committed counts of open tickets missing each field, and with the open tickets among the device-matched tickets in `fu/fu_ticket_last10.csv`.

In [6]:
last10 = (R.read_csv(OUT, 'fu/fu_ticket_last10.csv',
                     usecols=['case_id', 'opened_at_eastern', 'status', 'trouble_code', 'category'])
          .drop_duplicates('case_id'))  # the table has one row per ticket and window


def filled(group, field):
    """Tickets in a group of ts_field_fill_rates.csv that carry the field: the group's share times its tickets."""
    count = fill.loc[group, field] * fill.loc[group, 'tickets']
    assert abs(count - round(count)) < 0.25, (group, field, count)
    return int(round(count))


OPEN = 'open (Status=Dispatch)'
n_open = int(fill.loc[OPEN, 'tickets'])
open_code, open_category = filled(OPEN, 'Trouble Code'), filled(OPEN, 'Trouble Code Category')

# Cross-checks: timestamp_semantics.py's own count, audit_case_logic.py's committed tables, device-matched tickets
assert open_code == int(fact['C', 'open_tickets_with_trouble_code'])
groups = R.read_csv(OUT, 'audit_case_service_group_by_status.csv').set_index('service_group')['Dispatch']
assert groups['Missing trouble code'] + groups['Other recorded issue'] == groups['All']  # the only groups of open tickets
no_category = R.read_csv(OUT, 'audit_case_missing_category_composition.csv').query("status == 'Dispatch'")
assert set(no_category.trouble_code) == {'[Missing]'}  # open tickets without a Category also lack a Trouble Code
matched_open = last10[last10.status == 'Dispatch']
open_table = pd.DataFrame(
    [[n_open, open_code, open_category],
     [groups['All'], groups['Other recorded issue'], groups['All'] - no_category.cases.sum()],
     [len(matched_open), matched_open.trouble_code.notna().sum(), matched_open.category.notna().sum()]],
    index=['ts_field_fill_rates.csv', 'audit_case_*.csv (committed)', 'fu/fu_ticket_last10.csv (device-matched)'],
    columns=['open tickets', 'with a Trouble Code', 'with a Category']).astype(int)
assert open_table.iloc[0].equals(open_table.iloc[1])
print(f'Trouble Code is filled on {open_code} and Category on {open_category} of the {n_open} open tickets.')
open_table

Trouble Code is filled on 1 and Category on 1 of the 64 open tickets.


,open tickets,with a Trouble Code,with a Category
ts_field_fill_rates.csv,64,1,1
audit_case_*.csv (committed),64,1,1
fu/fu_ticket_last10.csv (device-matched),56,1,1


## The comparison month

`ts_facts.csv` lists the open tickets by opening month, and all 64 were opened in August 2026, so we compare them with the tickets opened that month that were already closed.

In [7]:
import ast

open_by_month = ast.literal_eval(fact['C', 'open_tickets_by_month'])
assert sum(open_by_month.values()) == n_open and len(open_by_month) == 1
(month,) = open_by_month
month_label = pd.Period(month, freq='M').strftime('%B %Y')
print(f'All {n_open} open tickets were opened in {month_label}.')

All 64 open tickets were opened in August 2026.


## August 2026 tickets with status Trouble Closed

We take the tickets opened in August 2026 from `ts_text_markers_by_month.csv` and subtract those with the other four statuses, which `ts_facts.csv` lists by opening month (Dispatch, Ricoh-Closed, Closed, and Case Closed). "Of any kind" means that every record type and service group counts, planned work included. We cross-check the counts with `audit_case_logic.py`'s committed table of statuses by opening month.

In [8]:
status_counts = ast.literal_eval(fact['B', 'status_counts'])
other_statuses = {'Dispatch': open_by_month, **ast.literal_eval(fact['C', 'other_closed_statuses_by_month'])}
assert set(other_statuses) | {'Trouble Closed'} == set(status_counts)
assert all(sum(per_month.values()) == status_counts[s] for s, per_month in other_statuses.items())
in_month = {s: per_month.get(month, 0) for s, per_month in other_statuses.items()}
in_month = {'Trouble Closed': int(tickets_by_month[month]) - sum(in_month.values()), **in_month}
n_aug_closed = in_month['Trouble Closed']

audit_month = R.read_csv(OUT, 'audit_case_status_by_month.csv').set_index('month').loc[month]
status_table = pd.DataFrame({'ts_text_markers_by_month.csv, ts_facts.csv': pd.Series(in_month),
                             'audit_case_status_by_month.csv (committed)': audit_month}).reindex(list(in_month))
status_table = status_table.astype(int)
status_table.loc['all statuses'] = status_table.sum()
assert (status_table.iloc[:, 0] == status_table.iloc[:, 1]).all()
print(f'{n_aug_closed} of the {tickets_by_month[month]} tickets opened in {month_label} have status Trouble Closed.')
status_table

461 of the 541 tickets opened in August 2026 have status Trouble Closed.


,"ts_text_markers_by_month.csv, ts_facts.csv",audit_case_status_by_month.csv (committed)
Trouble Closed,461,461
Dispatch,64,64
Ricoh-Closed,11,11
Closed,1,1
Case Closed,4,4
all statuses,541,541


## Trouble Code and Category on the 461 ("nearly all")

`ts_field_fill_rates.csv` gives fill shares for Trouble Closed tickets by age at export, counted back from the latest timestamp in the export (Sep 1st 08:05), and its under-7-day and 7-to-30-day groups hold 460 of the 461, all opened in August; we turn their shares into counts as above. The other ticket was opened on Aug 1st, and we read its two fields in the per-ticket table `fu/fu_ticket_last10.csv`. The report's "nearly all" rests on these counts, which docs/04 (§1 and §5) gives as 461 of 461 with a Trouble Code and 99.3% with a Category, that is, 458 tickets.

In [9]:
def day(ts):
    """A date as 'Aug 2nd'."""
    suffix = 'th' if 11 <= ts.day <= 13 else {1: 'st', 2: 'nd', 3: 'rd'}.get(ts.day % 10, 'th')
    return f'{ts:%b} {ts.day}{suffix}'


def when(ts):
    """A timestamp as 'Aug 2nd 08:05'."""
    return f'{day(ts)} {ts:%H:%M}'


export_ref = pd.Timestamp(fact['load', 'latest_timestamp_any_column (export ran at/after)'])
cut_7, cut_30 = export_ref - pd.Timedelta(days=7), export_ref - pd.Timedelta(days=30)
# Every ticket aged under 30 days at export was opened in the comparison month
assert cut_30.strftime('%Y-%m') == month == pd.Timestamp(fact['load', 'last_opened']).strftime('%Y-%m')
age_groups = [(f'after {when(cut_7)} (under 7 days)', 'age <7d Trouble Closed only'),
              (f'after {when(cut_30)}, up to {when(cut_7)} (7 to 30 days)', 'age 7-30d Trouble Closed only')]
rows = [(label, int(fill.loc[group, 'tickets']), filled(group, 'Trouble Code'), filled(group, 'Trouble Code Category'))
        for label, group in age_groups]

# The rest were opened earlier in the month (30 days or more before the export); the per-ticket table holds them all
opened = pd.to_datetime(last10.opened_at_eastern)
in_comparison = (last10.status == 'Trouble Closed') & (opened.dt.strftime('%Y-%m') == month)
older = last10[in_comparison & (opened <= cut_30)]
assert len(older) == n_aug_closed - sum(row[1] for row in rows)
first_day = pd.Period(month, freq='M').start_time
rows.append((f'{day(first_day)} up to {when(cut_30)} (30 days or more)', len(older),
             int(older.trouble_code.notna().sum()), int(older.category.notna().sum())))

nearly = pd.DataFrame(rows, columns=['Trouble Closed tickets opened', 'tickets', 'with a Trouble Code',
                                     'with a Category']).set_index('Trouble Closed tickets opened')
nearly.loc[f'all of {month_label}'] = nearly.sum()
assert nearly.loc[f'all of {month_label}', 'tickets'] == n_aug_closed
aug_code = int(nearly.loc[f'all of {month_label}', 'with a Trouble Code'])
aug_category = int(nearly.loc[f'all of {month_label}', 'with a Category'])
aug_category_share = aug_category / n_aug_closed

matched_aug = last10[in_comparison]
print(f'{aug_code} of the {n_aug_closed} have a Trouble Code and {aug_category} ({aug_category_share:.1%}) a Category. '
      f'Per ticket, the {len(matched_aug)} device-matched ones in fu/fu_ticket_last10.csv include '
      f'{matched_aug.trouble_code.notna().sum()} with a Trouble Code and {matched_aug.category.notna().sum()} with a '
      f'Category.')
nearly

461 of the 461 have a Trouble Code and 458 (99.3%) a Category. Per ticket, the 401 device-matched ones in fu/fu_ticket_last10.csv include 401 with a Trouble Code and 398 with a Category.


,tickets,with a Trouble Code,with a Category
Trouble Closed tickets opened,,,
after Aug 25th 08:05 (under 7 days),51,51,51
"after Aug 2nd 08:05, up to Aug 25th 08:05 (7 to 30 days)",409,409,406
Aug 1st up to Aug 2nd 08:05 (30 days or more),1,1,1
all of August 2026,461,461,458


## Category set near close, and final values only

IPA's answers at the Oct 13th meeting were that Trouble Code is entered at intake and may be updated by close, so the export holds only its final value, and that Category and Sub Category are set near the end of a ticket's life (docs/04, update of Oct 14th, and question 3 in §6). These are records of IPA's answers, so the check has no row for them; the counts above fit them, with both fields on 1 of the 64 open tickets vs. nearly all of the closed ones from the same month.

## Check against the report

The check leaves out the lower end of the range of backward steps on hospital-local clocks (1,205), as explained above.

In [10]:
R.check([
    ('Steps between consecutive tickets in case-number order', '7,470', R.num(n_steps)),
    ('Backward steps on the export clock', '0', R.num(export.backward)),
    ('Backward steps on hospital-local clocks, upper end', '1,312', R.num(local.backward.max())),
    ('Open tickets (status Dispatch)', '64', R.num(n_open)),
    ('Open tickets with a Trouble Code', '1', R.num(open_code)),
    ('Open tickets with a Category', '1', R.num(open_category)),
    ('Month compared (all open tickets were opened in it)', 'August 2026', month_label),
    ('Tickets opened that month with status Trouble Closed', '461', R.num(n_aug_closed)),
    ('"Nearly all": of the 461, with a Trouble Code', '461', R.num(aug_code)),
    ('"Nearly all": of the 461, with a Category', '458', R.num(aug_category)),
    ('"Nearly all": share of the 461 with a Category', '99.3%', R.pct(aug_category_share, 1)),
]);

,Finding,Report,Recomputed,Match
0,Steps between consecutive tickets in case-numb...,"7,470","7,470",yes
1,Backward steps on the export clock,0,0,yes
2,"Backward steps on hospital-local clocks, upper...","1,312","1,312",yes
3,Open tickets (status Dispatch),64,64,yes
4,Open tickets with a Trouble Code,1,1,yes
5,Open tickets with a Category,1,1,yes
6,Month compared (all open tickets were opened i...,August 2026,August 2026,yes
7,Tickets opened that month with status Trouble ...,461,461,yes
8,"""Nearly all"": of the 461, with a Trouble Code",461,461,yes
9,"""Nearly all"": of the 461, with a Category",458,458,yes


All 11 values match the report.
